# T27. p-mediana y cobertura máxima origen–destino

**Objetivo.** Comparar una red que reduce la distancia de acceso con una que maximiza viajes completos cubiertos. El voraz de extremos se recalcula con los candidatos de T24 como referencia común.

La p-mediana pondera las celdas por sus extremos observados. El modelo representa las 16 asignaciones más cercanas y acota las restantes mediante el coste de la siguiente estación. La red se evalúa con todas las distancias geográficas. Para MCLP se usa la cobertura agrupada de T26 y se fija el número de estaciones. El barrido toma las anclas de T13 (80, 90 y 95 % de extremos por área), el presupuesto de T26 y una estación menos.

In [1]:
import numpy as np
import pandas as pd
from IPython.display import display

from comun.config import ruta
from redes import (CoberturaOD, Problema, RADIOS, cobertura_split, distancias, guardar, leer,
                   pmediana, semillas_pmediana)

LIMITE_S = 5
p = Problema.del_ajuste()
minimos = leer("T26_resultados").set_index("radio_m")["k"]
T13 = pd.read_csv(ruta("tablas") / "exploracion" / "T13_estaciones_orden_magnitud.csv")
ks = {}
for radio in RADIOS:
    k = int(minimos.loc[radio])
    anclas = T13.loc[(T13["radio_m"] == radio) & (T13["cuota_pct"].isin([80, 90, 95])),
                    "estaciones_orden_magnitud"]
    ks[radio] = sorted(k for k in {*anclas, k, k - 1} if 0 < k <= p.n)

d_celdas = distancias(p.candidatos[["lon", "lat"]], p.candidatos[["lon", "lat"]])
pesos = p.candidatos["extremos"].to_numpy()
base_mediana = semillas_pmediana(d_celdas, pesos, max(max(v) for v in ks.values()))
redes25 = leer("T25_redes")
redes26 = leer("T26_redes")
lscp = leer("T26_resultados").set_index("radio_m")

medianas = {}
for k in sorted(set().union(*ks.values())):
    bases = [base_mediana[:k]]
    if f"kmedoids_k{k}" in set(redes25.loc[redes25["k"].eq(k), "network_id"]):
        bases.append(p.indices(redes25.loc[redes25["network_id"].eq(f"kmedoids_k{k}"), "station_id"]))
    semilla = min(bases, key=lambda ids: d_celdas[:, ids].min(axis=1) @ pesos)
    medianas[k] = pmediana(d_celdas, pesos, semilla, LIMITE_S)
    print(f"p-mediana k={k}", flush=True)

resultados, redes, excluidos = [], [], []
for radio in RADIOS:
    od = CoberturaOD(p, radio)
    prefijo, _ = od.voraz()
    id_ls = lscp.loc[radio, "network_id"]
    ls = p.indices(redes26.loc[redes26["network_id"].eq(id_ls), "station_id"])
    for k in ks[radio]:
        semillas = [prefijo[:k], medianas[k][0]]
        if len(ls) <= k:
            resto = prefijo[~np.isin(prefijo, ls)][:k - len(ls)]
            semillas.append(np.r_[ls, resto])
        semilla = od.intercambiar(max(semillas, key=od.cubiertos))
        op, calidad = od.resolver(semilla, LIMITE_S)
        for metodo, indices, info in (
            ("voraz", prefijo[:k], {"origen_solucion": "voraz extremos"}),
            ("pmediana", *medianas[k]),
            ("mclp", op, calidad),
        ):
            red_id = f"{metodo}_r{radio}_k{k}"
            medidas = p.medir(indices, radio)
            resultados.append({"network_id": red_id, "metodo": metodo, "radio_m": radio, "k": k,
                               **medidas, **info})
            redes.append(p.estaciones(indices, red_id))
            if metodo == "mclp":
                o, d, ambos = cobertura_split(p.distancia[:, indices].min(axis=1), radio)
                excluidos.append(pd.DataFrame({
                    "network_id": red_id, "trip_id": p.viajes.loc[~ambos, "trip_id"],
                    "fuera_origen": (~o)[~ambos], "fuera_destino": (~d)[~ambos],
                }))
        print(f"MCLP r={radio} k={k}", flush=True)

resumen = pd.DataFrame(resultados)
guardar(resumen, "T27_resultados")
guardar(pd.concat(redes, ignore_index=True), "T27_redes")
archivo = ruta("viajes_limpios").with_name("T27_viajes_excluidos.parquet")
pd.concat(excluidos, ignore_index=True).to_parquet(archivo, index=False)
display(resumen.loc[resumen["metodo"].ne("voraz"),
                    ["radio_m", "k", "metodo", "pct_viajes_cubiertos", "d_p90_m",
                     "brecha_pct", "origen_solucion"]].round(3))

p-mediana k=58


p-mediana k=80


p-mediana k=85


p-mediana k=86


p-mediana k=97


p-mediana k=102

... (31 outputs left out)

MCLP r=400 k=85


MCLP r=400 k=86


MCLP r=400 k=97


,radio_m,k,metodo,pct_viajes_cubiertos,d_p90_m,brecha_pct,origen_solucion
1,200,230,pmediana,66.842,291.730,0.000,MILP
2,200,230,mclp,70.196,356.672,0.000,MILP
4,200,288,pmediana,78.259,220.279,0.000,MILP
5,200,288,mclp,79.959,220.551,0.000,MILP
7,200,289,pmediana,78.455,219.022,0.000,MILP
8,200,289,mclp,80.103,219.217,0.000,MILP
10,200,317,pmediana,82.615,199.021,0.000,MILP
11,200,317,mclp,83.779,198.641,0.000,MILP
13,200,387,pmediana,90.445,186.280,0.000,MILP
14,200,387,mclp,91.060,186.649,0.000,MILP


**Lectura.** Las cotas de MCLP se expresan en viajes completos y las de p-mediana en metros medios por extremo de celda. La cobertura y los cuantiles que comparan redes se calculan con las coordenadas de cada viaje, sobre todos los extremos del ajuste. El Parquet local conserva, por red MCLP, qué viaje queda fuera y en qué extremos.

**Comparación y evidencia.** Se mantienen las redes archivadas. Para cada radio y presupuesto se comparan las redes sobre los mismos viajes y candidatos T24. MCLP maximiza el número de viajes completos. P-mediana minimiza la media ponderada de distancia de los extremos agrupados en H3, cuyos pesos equivalen a su frecuencia. Sus objetivos tienen unidades distintas. El análisis compara ambas soluciones por viajes con cero, uno y dos extremos cubiertos y por P90 de las distancias de acceso, también separado por origen y destino. La auditoría de `certificados.py` acota desde arriba el máximo de viajes de MCLP mediante un dual entero verificado. `T27_certificados.csv` da el intervalo y su amplitud en viajes de ajuste para cada presupuesto.